# Stage 4: Inferencja na zbiorze testowym i zapis predykcji

Ten notatnik:
1. Wczytuje model po fine-tuningu (etap 3.5).
2. Wczytuje testowy plik `.parquet` (`mol_id`, `SMILES`).
3. Robi predykcje `class_0..class_499` jako 0/1.
4. Zapisuje wynikowy plik `.parquet` w wymaganym formacie.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
# Sciezki
cwd = Path.cwd()
ONTOLOGY_DIR = cwd / "1_ontology" if (cwd / "1_ontology").exists() else cwd
DATA_DIR = ONTOLOGY_DIR / "data"
ART2_DIR = DATA_DIR / "stage2_artifacts"
ART35_DIR = DATA_DIR / "stage3_5_artifacts"

TEST_PATH = DATA_DIR / "chebi_dataset_test_empty.parquet"
OUT_PATH = DATA_DIR / "task1_submission.parquet"

META_PATH = ART2_DIR / "stage2_fingerprints_meta.json"
NPZ_PATH = ART2_DIR / "stage2_fingerprints.npz"

MODEL_PATH = ART35_DIR / "hier_gnn_finetuned.pt"
REFINER_PATH = ART35_DIR / "logit_refiner_finetuned.pt"
THRESHOLDS_PATH = ART35_DIR / "class_thresholds.json"

for p in [TEST_PATH, META_PATH, NPZ_PATH, MODEL_PATH, REFINER_PATH, THRESHOLDS_PATH]:
    print(p, "OK" if p.exists() else "MISSING")

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\chebi_dataset_test_empty.parquet OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints_meta.json OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints.npz OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\hier_gnn_finetuned.pt OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\logit_refiner_finetuned.pt OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\class_thresholds.json OK


In [3]:
# Wczytanie danych testowych i metadanych klas
test_df = pd.read_parquet(TEST_PATH)
required_cols = {"mol_id", "SMILES"}
if not required_cols.issubset(test_df.columns):
    raise ValueError(f"Plik testowy musi zawierac kolumny: {required_cols}")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
class_cols = meta.get("class_columns", [f"class_{i}" for i in range(500)])
if len(class_cols) != 500:
    raise ValueError(f"Oczekiwano 500 klas, otrzymano: {len(class_cols)}")

npz = np.load(NPZ_PATH)
M_ancestor_np = npz["M_ancestor"].astype(np.uint8)

print("Test rows:", len(test_df))
print("N classes:", len(class_cols))

Test rows: 11223
N classes: 500


In [4]:
# Definicje modeli (zgodne z etapem 3 / 3.5)
class HierGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, out_dim: int = 500, dropout: float = 0.2):
        super().__init__()
        self.mlp1 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.mlp2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(self.mlp1)
        self.conv2 = GINConv(self.mlp2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        return self.head(x)


class LogitRefiner(nn.Module):
    def __init__(self, n_classes: int = 500, hidden1: int = 1024, hidden2: int = 512, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_classes, hidden1),
            nn.LayerNorm(hidden1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.LayerNorm(hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, n_classes),
        )

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return self.net(logits)


class LogitRefinerLegacy(nn.Module):
    # Legacy architecture from older stage 3.5 checkpoints: 500 -> hidden -> 500.
    def __init__(self, n_classes: int = 500, hidden: int = 1024, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_classes, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return self.net(logits)


def build_refiner_from_state_dict(state_dict: dict, n_classes: int = 500) -> nn.Module:
    # New architecture has final layer key net.8.weight, legacy has net.3.weight.
    if "net.8.weight" in state_dict:
        hidden1 = int(state_dict["net.0.weight"].shape[0])
        hidden2 = int(state_dict["net.4.weight"].shape[0])
        return LogitRefiner(n_classes=n_classes, hidden1=hidden1, hidden2=hidden2, dropout=0.2)
    if "net.3.weight" in state_dict:
        hidden = int(state_dict["net.0.weight"].shape[0])
        return LogitRefinerLegacy(n_classes=n_classes, hidden=hidden, dropout=0.2)
    raise RuntimeError("Nieznany format checkpointu refiner (brak net.8.weight i net.3.weight).")

In [5]:
# Budowa grafow z SMILES (uproszczone cechy atomowe jak w etapach 3/3.5)
def atom_features(atom: Chem.Atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
    ]


def smiles_to_data(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    edges = []
    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()
        edges.append([i, j])
        edges.append([j, i])

    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index)


graphs = []
bad_idx = []
for i, smi in enumerate(test_df["SMILES"].astype(str).tolist()):
    data = smiles_to_data(smi)
    if data is None:
        bad_idx.append(i)
        continue
    data.sample_idx = i
    graphs.append(data)

print("Poprawne grafy:", len(graphs))
print("Bledne SMILES:", len(bad_idx))
loader = DataLoader(graphs, batch_size=256, shuffle=False)

[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] Unusual charge on atom 6 number of radical electrons set to zero
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:35] WARNING: not removing hydrogen atom without neighbors
[20:08:36] WARNING: not removing hydrogen atom without neighbors
[20:08:36] WAR

Poprawne grafy: 11223
Bledne SMILES: 0


In [6]:
# Zaladowanie modelu i inferencja
if len(graphs) == 0:
    raise RuntimeError("Brak poprawnych SMILES do inferencji.")

in_dim = graphs[0].x.shape[1]
model = HierGNN(in_dim=in_dim, hidden_dim=128, out_dim=500, dropout=0.2).to(device)

ckpt_model = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt_model["model_state_dict"])

ckpt_ref = torch.load(REFINER_PATH, map_location=device)
refiner_state = ckpt_ref["refiner_state_dict"]
refiner = build_refiner_from_state_dict(refiner_state, n_classes=500).to(device)
refiner.load_state_dict(refiner_state)
print("Wczytano refiner type:", refiner.__class__.__name__)

model.eval()
refiner.eval()

probs_valid = []
idx_valid = []
with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        logits = refiner(logits)
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_valid.append(probs)
        idx_valid.extend(batch.sample_idx.cpu().numpy().tolist())

probs_valid = np.vstack(probs_valid) if probs_valid else np.empty((0, 500), dtype=np.float32)
idx_valid = np.array(idx_valid, dtype=np.int64)

print("Predykcje dla poprawnych grafow:", probs_valid.shape)

Wczytano refiner type: LogitRefinerLegacy
Predykcje dla poprawnych grafow: (11223, 500)


In [7]:
# Progowanie (global + per-class) i closure hierarchiczne
thresholds_payload = json.loads(THRESHOLDS_PATH.read_text(encoding="utf-8"))
global_t = float(thresholds_payload.get("global_threshold", ckpt_model.get("global_threshold", 0.5)))

class_thr_map = thresholds_payload.get("class_thresholds", {})
class_thresholds = np.full((500,), global_t, dtype=np.float32)
for i, c in enumerate(class_cols):
    if c in class_thr_map:
        class_thresholds[i] = float(class_thr_map[c])

probs_all = np.zeros((len(test_df), 500), dtype=np.float32)
if len(idx_valid) > 0:
    probs_all[idx_valid] = probs_valid

pred_bin = (probs_all >= class_thresholds.reshape(1, -1)).astype(np.uint8)

def apply_closure(pred_bin_arr: np.ndarray, m_ancestor: np.ndarray) -> np.ndarray:
    pred = pred_bin_arr.copy()
    for child in range(pred.shape[1]):
        anc = np.where(m_ancestor[child] == 1)[0]
        if len(anc) == 0:
            continue
        rows = pred[:, child] == 1
        pred[np.ix_(rows, anc)] = 1
    return pred

pred_bin = apply_closure(pred_bin, M_ancestor_np)
print("Gotowe predykcje 0/1:", pred_bin.shape)

Gotowe predykcje 0/1: (11223, 500)


In [8]:
# Zapis finalnego pliku parquet
out_df = test_df[["mol_id", "SMILES"]].copy()
for i, c in enumerate(class_cols):
    out_df[c] = pred_bin[:, i].astype(np.uint8)

out_df.to_parquet(OUT_PATH, index=False)

print("Zapisano:", OUT_PATH)
print("Shape:", out_df.shape)
print("Podglad:")
out_df.head(3)

C:\Users\ratch\AppData\Local\Temp\ipykernel_49416\661882852.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[c] = pred_bin[:, i].astype(np.uint8)
C:\Users\ratch\AppData\Local\Temp\ipykernel_49416\661882852.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[c] = pred_bin[:, i].astype(np.uint8)
C:\Users\ratch\AppData\Local\Temp\ipykernel_49416\661882852.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

Zapisano: c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\task1_submission.parquet
Shape: (11223, 502)
Podglad:


,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...,1,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
# Szybka walidacja formatu
check_df = pd.read_parquet(OUT_PATH)
expected_cols = ["mol_id", "SMILES"] + [f"class_{i}" for i in range(500)]
assert list(check_df.columns) == expected_cols, "Niepoprawny zestaw kolumn w pliku wynikowym"

vals = np.unique(check_df[[f"class_{i}" for i in range(500)]].to_numpy())
assert set(vals.tolist()).issubset({0, 1}), f"Wykryto wartosci inne niz 0/1: {vals}"

print("Format OK. Kolumny i wartosci sa poprawne.")

Format OK. Kolumny i wartosci sa poprawne.
